In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
blocks = catalog.load_table("raw.blocks")
blocks.manage_snapshots().create_tag(blocks.current_snapshot().snapshot_id, "after-cleanup").commit()
stats = catalog.load_table("ref.daily_block_stats")
stats.manage_snapshots().create_branch(stats.current_snapshot().snapshot_id, "audit").commit()
[(r["name"], r["type"]) for r in stats.inspect.refs().to_pylist()]

[('main', 'BRANCH'), ('audit', 'BRANCH')]

In [3]:
november = con.execute(f"""
    select date as block_date, count(*) as blocks, sum(transaction_count) as transactions,
           round(avg(100.0 * gas_used / gas_limit), 2) as avg_gas_used_pct,
           round(avg(base_fee_per_gas) / 1e9, 4) as avg_base_fee_gwei
    from read_parquet('{S3}/blocks/date=2026-11-0[123]/*.parquet')
    group by 1 order by 1
""").to_arrow_table()
stats.append(november.cast(stats.schema().as_arrow()), branch="audit")

In [4]:
audit = stats.scan(snapshot_id=stats.snapshot_by_name("audit").snapshot_id).to_duckdb("audit")
audit.sql("select count(*) as days, count(distinct block_date) as distinct_days, min(blocks) as min_blocks from audit")

┌───────┬───────────────┬────────────┐
│ days  │ distinct_days │ min_blocks │
│ int64 │     int64     │   int64    │
├───────┼───────────────┼────────────┤
│    95 │            95 │       7138 │
└───────┴───────────────┴────────────┘

In [5]:
stats.scan().to_arrow().num_rows

92

In [6]:
stats.manage_snapshots().set_current_snapshot(snapshot_id=stats.snapshot_by_name("audit").snapshot_id).commit()
stats.scan().to_arrow().num_rows

95